
# Core Notebook 4 · Cashflows, XIRR, and Numerical Methods

Cashflows are modeled as typed events, and the math module exposes deterministic solvers and integration utilities that avoid hand-rolled numerical code.


## Imports

In [ ]:

from datetime import date

from finstack.core import cashflow, money, currency
from finstack.core.math import solver, integration, distributions

usd = currency.Currency("USD")


## Cashflow primitives

In [ ]:

coupon_cf = cashflow.CashFlow(
    date(2025, 4, 1),
    money.Money(5_000, usd),
    cashflow.CFKind.FIXED,
    accrual_factor=0.25,
)
coupon_cf.validate()
print("Cashflow tuple:", coupon_cf.to_tuple())

floating_cf = cashflow.CashFlow(
    date(2025, 4, 1),
    money.Money(4_750, usd),
    cashflow.CFKind.FLOAT_RESET,
    accrual_factor=0.247,
    reset_date=date(2025, 1, 1),
)
print("Floating kind:", floating_cf.kind.name)


## XIRR helper

In [ ]:

investment_flows = [
    (date(2024, 1, 1), -250_000.0),
    (date(2024, 6, 30), 35_000.0),
    (date(2024, 12, 31), 40_000.0),
    (date(2025, 6, 30), 45_000.0),
    (date(2025, 12, 31), 265_000.0),
]
irr = cashflow.xirr(investment_flows)
conservative_irr = cashflow.xirr(investment_flows, guess=0.05)
print(f"Implied XIRR: {irr:.6f}")
print(f"XIRR with 5% initial guess: {conservative_irr:.6f}")


## Numerical solvers

In [ ]:

price_target = 102.50
coupon_rate = 0.045
frequency = 2
periods = 8
face = 100.0

def bond_price_error(yield_guess: float) -> float:
    pv = 0.0
    for n in range(1, periods + 1):
        coupon_cash = face * coupon_rate / frequency
        pv += coupon_cash / (1 + yield_guess / frequency) ** n
    pv += face / (1 + yield_guess / frequency) ** periods
    return pv - price_target

brent = solver.BrentSolver(
    tolerance=1e-10,
    max_iterations=256,
    bracket_expansion=2.0,
    initial_bracket_size=0.01,
)
yield_solution = brent.solve(bond_price_error, 0.04)

newton = solver.NewtonSolver(
    tolerance=1e-10,
    max_iterations=32,
    fd_step=1e-6,
)
yield_newton = newton.solve(bond_price_error, 0.04)

print(f"Brent yield: {yield_solution:.6f}")
print(f"Newton yield: {yield_newton:.6f}")


## Integration utilities

In [ ]:

simpson_area = integration.simpson_rule(lambda t: t * t, 0.0, 1.0, intervals=24)
adaptive_area = integration.adaptive_simpson(lambda t: t * t, 0.0, 1.0, tol=1e-8, max_depth=12)
print("Simpson area:", simpson_area)
print("Adaptive Simpson area:", adaptive_area)


## Distribution helpers

In [ ]:

p = distributions.binomial_probability(trials=10, successes=3, probability=0.4)
log_nCk = distributions.log_binomial_coefficient(10, 3)
print("Binomial probability P(X=3):", p)
print("log(n choose k):", log_nCk)



## Checklist · best practices

- Model deterministic legs with `CashFlow` + `Money` instead of raw tuples
- Use `cashflow.xirr` before hand-rolling Newton/Bisection IRR solvers
- Keep solver tolerances/iteration caps explicit for reproducibility
- Reach for the integration helpers when valuing continuous cash streams or density-weighted payoffs
